# Exercise 1: Transformers Library Essentials

**Goal:** Load a pretrained model + tokenizer from HuggingFace, understand the tokenization flow, run inference, and experiment with generation parameters.

## Install Dependencies

In [2]:
!pip install -q transformers accelerate

## Imports



In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

PyTorch version: 2.10.0+cu128
CUDA available: True
Using device: cuda


## Load tokenizer and model

We use `gpt2` — small (124M params), fast to download, no auth needed.

In [4]:
MODEL_NAME = 'gpt2'

print(f'Loading tokenizer for {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 has no pad token by default; set it to eos_token
tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model for {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print(f'\nModel loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading tokenizer for gpt2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model loaded! Parameters: 124,439,808


## Understanding the tokenization flow

**Flow:** `text → tokens → token IDs → back to text`

In [12]:
text = 'KDU generates one of the best engineering talents in Kickdrum'

# Step 1: text → token IDs
encoding = tokenizer(text, return_tensors='pt')
input_ids = encoding['input_ids']

# Step 2: token IDs → individual token strings
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

print('Original text:')
print(' ', text)

print('\nToken IDs:')
print(' ', input_ids[0].tolist())

print('\nTokens (subwords):')
print(' ', tokens)

print('\nDecoded back to text:')
print(' ', tokenizer.decode(input_ids[0]))

print(f'\nVocab size: {tokenizer.vocab_size:,}')
print(f'Number of tokens in this sentence: {len(tokens)}')

Original text:
  KDU generates one of the best engineering talents in Kickdrum

Token IDs:
  [42, 35, 52, 18616, 530, 286, 262, 1266, 8705, 18054, 287, 10279, 67, 6582]

Tokens (subwords):
  ['K', 'D', 'U', 'Ġgenerates', 'Ġone', 'Ġof', 'Ġthe', 'Ġbest', 'Ġengineering', 'Ġtalents', 'Ġin', 'ĠKick', 'd', 'rum']

Decoded back to text:
  KDU generates one of the best engineering talents in Kickdrum

Vocab size: 50,257
Number of tokens in this sentence: 14


## Inference helper function

In [6]:
def generate_text(prompt, max_new_tokens=60, temperature=1.0, top_p=1.0, do_sample=True):
    """
    Run inference and return the generated text.

    Args:
        prompt          : Input text string
        max_new_tokens  : Number of NEW tokens to generate (not counting the prompt)
        temperature     : >1 = more random, <1 = more focused, 1 = neutral
        top_p           : Nucleus sampling threshold (0–1); 1.0 = disabled
        do_sample       : True = sampling, False = greedy decoding
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(new_ids, skip_special_tokens=True)
    return generated

print('Helper function ready.')

Helper function ready.


## Experiment: effect of temperature

**Temperature** controls how "sharp" the probability distribution is over next tokens:
- Low (0.3) → model picks the most likely tokens → repetitive but focused
- High (1.5) → distribution is flatter → more surprising and diverse

In [14]:
prompt = 'Once upon a time in a land far away in mountains,'

temperatures = [0.3, 0.7, 1.0, 1.5]

print(f'Prompt: "{prompt}"\n')
print('=' * 60)

for temp in temperatures:
    result = generate_text(prompt, max_new_tokens=50, temperature=temp, top_p=1.0)
    print(f'\nTemperature = {temp}')
    print(f'  {result}')
    print('-' * 60)

Prompt: "Once upon a time in a land far away in mountains,"


Temperature = 0.3
   the sun rises and sets in the sky. The sun is the sun, and the moon is the moon. The moon is the moon. The sun is the sun, and the moon is the moon. The sun is the sun, and the moon
------------------------------------------------------------

Temperature = 0.7
   people would run, climb and ride a mountain, be chased by wolves.

The wolves would come and chase you across the plains, where you'd never see them before. They would have their own people to look after them. So, as
------------------------------------------------------------

Temperature = 1.0
   a king of nations came and invited those he named in the land who were not Christians. At this event he sent forth all the servants of God; for he said they would help him in all the rest; and he said they would all help him
------------------------------------------------------------

Temperature = 1.5
   it gave the enemy an advantage as though the

## Experiment: effect of top_p (nucleus sampling)

**top_p** keeps only the smallest set of tokens whose cumulative probability ≥ p:
- Low (0.5) → only high-probability tokens considered → safer, less varied
- High (0.95) → wide range of tokens allowed → more creative

In [15]:
prompt = 'The Engineer looked through the code and discovered'

top_p_values = [0.5, 0.8, 0.95, 1.0]

print(f'Prompt: "{prompt}"\n')
print('=' * 60)

for top_p in top_p_values:
    result = generate_text(prompt, max_new_tokens=50, temperature=0.9, top_p=top_p)
    print(f'\ntop_p = {top_p}')
    print(f'  {result}')
    print('-' * 60)

Prompt: "The Engineer looked through the code and discovered"


top_p = 0.5
   that the new code was being used by a different project.

The Engineer then ran the code through the same test suite and found that the new code was being used by another project.

The Engineer then ran the code through the same test
------------------------------------------------------------

top_p = 0.8
   a few bugs. They were a common problem in the past, but the code was different now.

The problem was that when the code was being run, it would always try to make some kind of error. If it got a bad error
------------------------------------------------------------

top_p = 0.95
   that the same problem could be resolved with the following code:

/* * @author Richard C. Smith */ @inline void start() { var s_m_new = new_int((int) 1); var s_m_old
------------------------------------------------------------

top_p = 1.0
   that a function was being called on a node that was still active.

As the function w

## Experiment: effect of max_new_tokens

Controls output length. More tokens ≠ better quality — model can start
repeating or going off-topic past a certain point.

In [16]:
prompt = 'Artificial Intelligence will'

token_counts = [20, 50, 100]

print(f'Prompt: "{prompt}"\n')
print('=' * 60)

for n in token_counts:
    result = generate_text(prompt, max_new_tokens=n, temperature=0.8, top_p=0.9)
    print(f'\nmax_new_tokens = {n}')
    print(f'  {result}')
    print('-' * 60)

Prompt: "Artificial Intelligence will"


max_new_tokens = 20
   not be a threat to Apple's iOS products, as evidenced by the fact that it has the lowest
------------------------------------------------------------

max_new_tokens = 50
   be the next big tech, but not just for artificial intelligence. As IBM's chief technology officer, James W. Hartley, told the International Business Times: "We are building a computer that's a little more powerful than humans and less powerful than
------------------------------------------------------------

max_new_tokens = 100
   be able to create a "supercomputer" for your business, rather than simply a "computer" for your organization.

The new company will be able to provide a "supercomputer" for your organization, rather than simply a "computer" for your organization.

The "supercomputer" will also allow your business to be able to create and store a large amount of data on a single device, rather than a huge number of machines.

The "supercompu

## Greedy decoding vs sampling

With `do_sample=False` (greedy), the model always picks the single
most probable next token — deterministic but often repetitive.

In [17]:
prompt = 'In the future, robots will'

print(f'Prompt: "{prompt}"\n')
print('=' * 60)

greedy = generate_text(prompt, max_new_tokens=50, do_sample=False)
print(f'\nGreedy decoding (do_sample=False):')
print(f'  {greedy}')

sampled = generate_text(prompt, max_new_tokens=50, temperature=0.9, top_p=0.9, do_sample=True)
print(f'\nSampling (temperature=0.9, top_p=0.9):')
print(f'  {sampled}')

Prompt: "In the future, robots will"


Greedy decoding (do_sample=False):
   be able to do things like take care of people, and they'll be able to do things like take care of people, and they'll be able to do things like take care of people, and they'll be able to do things like take care

Sampling (temperature=0.9, top_p=0.9):
   be used for human tasks, including shopping, shopping trips, education, and so on. The only problem is that there is no way to automate these tasks in a timely manner. Therefore, there is no point in making a robot-centric policy.


## Summary table of parameters

Run your own custom prompt below with any combination of settings.

In [18]:
# ✏️ Try your own prompt and parameters here
MY_PROMPT       = 'The best way to learn swimming is'
MY_TEMPERATURE  = 0.8    # float: 0.1 – 2.0
MY_TOP_P        = 0.9    # float: 0.0 – 1.0
MY_MAX_TOKENS   = 60     # int

output = generate_text(
    MY_PROMPT,
    max_new_tokens=MY_MAX_TOKENS,
    temperature=MY_TEMPERATURE,
    top_p=MY_TOP_P,
)

print(f'Prompt    : {MY_PROMPT}')
print(f'Temp      : {MY_TEMPERATURE}')
print(f'top_p     : {MY_TOP_P}')
print(f'Max tokens: {MY_MAX_TOKENS}')
print(f'\nOutput:')
print(f'  {output}')

Prompt    : The best way to learn swimming is
Temp      : 0.8
top_p     : 0.9
Max tokens: 60

Output:
   to watch the water.

If you're looking for a better way to learn swimming, it's a lot like the one we've covered in the past.

Now, let's break down the best ways to learn swimming.

What are the main techniques you use to learn swimming
